In [ ]:
# Install ROUGE scorer for label generation and GNN libraries
!pip install -q rouge-score torch-geometric sentence-transformers networkx

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.6 MB/s eta 0:00:00


SHORT SUMMARIES

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, BatchNorm
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
import json

# --- CONFIGURATION ---
INPUT_FILE = "case_1_graph.pkl"
GOLD_SUMMARY_FILE = "gold_summary_short.txt"
EMBED_MODEL = "all-MiniLM-L6-v2"
# We define a range to test instead of a fixed target
CANDIDATE_KS = [3,5,7,10]

# 1. Load Data
print(f"Loading Graph: {INPUT_FILE}")
with open(INPUT_FILE, "rb") as f:
    G = pickle.load(f)

nodes = list(G.nodes())
node_to_idx = {n: i for i, n in enumerate(nodes)}
idx_to_node = {i: n for i, n in enumerate(nodes)}
doc_sentences = [str(G.nodes[n].get("text", "")) for n in nodes]

print(f"Loading Short Gold Summary: {GOLD_SUMMARY_FILE}")
with open(GOLD_SUMMARY_FILE, "r", encoding="utf-8") as f:
    gold_text = f.read()

# 2. Generate Labels (Binary Classification for Training)
# For training, we still need a target to learn from.
# We'll use the "Top 5" Oracle sentences as the positive class for training stability.
TRAIN_TARGET_K = 5
print(f"Generating Binary Labels (Targeting Top {TRAIN_TARGET_K} for training)...")

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
sent_scores = []
for i, sent in enumerate(doc_sentences):
    s = scorer.score(gold_text, sent)
    sent_scores.append((i, s['rougeL'].fmeasure))

sent_scores.sort(key=lambda x: x[1], reverse=True)
top_k_indices = {i for i, _ in sent_scores[:TRAIN_TARGET_K]}

y_np = np.array([1.0 if i in top_k_indices else 0.0 for i in range(len(doc_sentences))])
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

# 3. Embeddings & Graph
print("Computing Embeddings...")
embedder = SentenceTransformer(EMBED_MODEL)
x = torch.tensor(embedder.encode(doc_sentences, convert_to_numpy=True), dtype=torch.float32)

src, dst = [], []
for u, v in G.edges():
    src.append(node_to_idx[u])
    dst.append(node_to_idx[v])
edge_index = torch.tensor([src, dst], dtype=torch.long)

# 4. Model (Classification)
class ShortGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=128):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)
        self.head = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.4, training=self.training)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.4, training=self.training)
        return self.head(x)

# 5. Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ShortGCN(in_channels=x.shape[1]).to(device)
x, edge_index, y = x.to(device), edge_index.to(device), y.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)
criterion = nn.BCEWithLogitsLoss()

print("Training Short GCN...")
model.train()
for epoch in range(300):
    optimizer.zero_grad()
    loss = criterion(model(x, edge_index), y)
    loss.backward()
    optimizer.step()

# 6. Inference & Dynamic K Selection
model.eval()
with torch.no_grad():
    logits = model(x, edge_index)
    probs = torch.sigmoid(logits).cpu().numpy().flatten()

# Prepare all candidates
full_results = []
for i, score in enumerate(probs):
    full_results.append({
        "node_index": i,
        "orig_node_id": idx_to_node[i],
        "score": float(score),
        "text": doc_sentences[i]
    })

# Sort by predicted probability (confidence)
full_results.sort(key=lambda x: x['score'], reverse=True)

# Loop through candidate K values to find the best one
best_k = TRAIN_TARGET_K
best_metric = -1.0

print(f"\nEvaluating Candidate K values: {CANDIDATE_KS}")
for k in CANDIDATE_KS:
    # Select Top K
    subset = full_results[:k]

    # Re-sort by document order for coherent reading
    subset.sort(key=lambda x: x['node_index'])

    # Generate summary text
    text_summary = " ".join([s['text'] for s in subset])

    # Evaluate against Gold Short Summary
    # Using ROUGE-L F-Measure as the deciding metric
    score = scorer.score(gold_text, text_summary)['rougeL'].fmeasure

    print(f"  K={k}: ROUGE-L={score:.4f}")

    if score > best_metric:
        best_metric = score
        best_k = k

print(f"✅ Optimal K found: {best_k} (Score: {best_metric:.4f})")

# Final Selection using Best K
selected_items = full_results[:best_k]
selected_items.sort(key=lambda x: x['node_index'])

# 7. Output JSON
output_data = {
    "input_file": INPUT_FILE,
    "method": "gcn_binary_classification_dynamic_k",
    "parameters": {
        "best_k": best_k,
        "candidate_k_range": CANDIDATE_KS,
        "gold_file_used": GOLD_SUMMARY_FILE,
        "rouge_l_score": best_metric
    },
    "selected": selected_items
}

with open("short_summary_output.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"Short summary saved to: short_summary_output.json")

Loading Graph: case_1_graph.pkl
Loading Short Gold Summary: gold_summary_short.txt
Generating Binary Labels (Targeting Top 5 for training)...
Computing Embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Training Short GCN...

Evaluating Candidate K values: [3, 5, 7, 10]
  K=3: ROUGE-L=0.1976
  K=5: ROUGE-L=0.1978
  K=7: ROUGE-L=0.2171
  K=10: ROUGE-L=0.1988
✅ Optimal K found: 7 (Score: 0.2171)
Short summary saved to: short_summary_output.json


LONG SUMMARIES

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, BatchNorm
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
import json

# --- CONFIGURATION ---
INPUT_FILE = "case_1_graph.pkl"
GOLD_SUMMARY_FILE = "gold_summary_long.txt" # <--- UPDATED: Specific Long Summary
EMBED_MODEL = "all-MiniLM-L6-v2"

# 1. Load Data
print(f"Loading Graph: {INPUT_FILE}")
with open(INPUT_FILE, "rb") as f:
    G = pickle.load(f)

nodes = list(G.nodes())
node_to_idx = {n: i for i, n in enumerate(nodes)}
idx_to_node = {i: n for i, n in enumerate(nodes)}
doc_sentences = [str(G.nodes[n].get("text", "")) for n in nodes]

print(f"Loading Long Gold Summary: {GOLD_SUMMARY_FILE}")
with open(GOLD_SUMMARY_FILE, "r", encoding="utf-8") as f:
    gold_text = f.read()

# 2. Generate Labels (Regression against LONG Gold)
print("Generating Regression Labels...")
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
scores = []
for sent in doc_sentences:
    # We use Recall here because long summaries are about coverage
    scores.append(scorer.score(gold_text, sent)['rougeL'].recall)

scores = np.array(scores)
if scores.max() > 0:
    y_np = (scores - scores.min()) / (scores.max() - scores.min())
else:
    y_np = scores
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

# 3. Embeddings & Graph
print("Computing Embeddings...")
embedder = SentenceTransformer(EMBED_MODEL)
x = torch.tensor(embedder.encode(doc_sentences, convert_to_numpy=True), dtype=torch.float32)

src, dst = [], []
for u, v in G.edges():
    src.append(node_to_idx[u])
    dst.append(node_to_idx[v])
edge_index = torch.tensor([src, dst], dtype=torch.long)

# 4. Model (Regression)
class LongGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=128):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)
        self.head = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        return torch.sigmoid(self.head(x))

# 5. Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LongGCN(in_channels=x.shape[1]).to(device)
x, edge_index, y = x.to(device), edge_index.to(device), y.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.MSELoss()

print("Training Long GCN...")
model.train()
for epoch in range(300):
    optimizer.zero_grad()
    loss = criterion(model(x, edge_index), y)
    loss.backward()
    optimizer.step()

# 6. Inference & Dynamic K
model.eval()
with torch.no_grad():
    scores_pred = model(x, edge_index).cpu().numpy().flatten()

full_results = []
for i, score in enumerate(scores_pred):
    full_results.append({
        "node_index": i,
        "orig_node_id": idx_to_node[i],
        "score": float(score),
        "text": doc_sentences[i]
    })

full_results.sort(key=lambda x: x['score'], reverse=True)

# Find Optimal K against LONG Gold
candidate_ks = [10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60]
best_k = 20
best_metrics = -1.0

for k in candidate_ks:
    subset = full_results[:k]
    subset.sort(key=lambda x: x['node_index'])
    text_summary = " ".join([s['text'] for s in subset])

    current_score = scorer.score(gold_text, text_summary)['rougeL'].fmeasure
    if current_score > best_metrics:
        best_metrics = current_score
        best_k = k

selected_items = full_results[:best_k]
selected_items.sort(key=lambda x: x['node_index'])

output_data = {
    "input_file": INPUT_FILE,
    "method": "gcn_regression_dynamic_k",
    "parameters": {
        "best_k": best_k,
        "gold_file_used": GOLD_SUMMARY_FILE,
        "rouge_l_score": best_metrics
    },
    "selected": selected_items
}

with open("long_summary_output.json", "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"Long summary saved to: long_summary_output.json")

Loading Graph: case_1_graph.pkl
Loading Long Gold Summary: gold_summary_long.txt
Generating Regression Labels...
Computing Embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training Long GCN...
Long summary saved to: long_summary_output.json


In [ ]:
# --- INFERENCE & DYNAMIC K COMPARISON (LONG SUMMARY) ---
model.eval()
with torch.no_grad():
    # Get regression scores (0.0 to 1.0)
    scores_pred = model(x, edge_index).cpu().numpy().flatten()

# 1. Prepare Full Results List
full_results = []
for i, score in enumerate(scores_pred):
    full_results.append({
        "node_index": i,
        "orig_node_id": idx_to_node[i],
        "score": float(score),
        "text": doc_sentences[i]
    })

# 2. Sort by Model Confidence (Importance Score)
full_results.sort(key=lambda x: x['score'], reverse=True)

# 3. Define Candidate K Values for Long Summaries
# We test a wider range since long summaries vary more in optimal length
CANDIDATE_KS = [10, 15, 20, 25, 30, 35, 40, 45, 50]

print(f"\n--- Evaluating Candidate K Values (Long Summary) ---")
print(f"{'K':<5} | {'ROUGE-L':<8} | {'Status'}")
print("-" * 45)

best_k = 20 # Default fallback
best_metric = -1.0

# 4. Evaluation Loop
for k in CANDIDATE_KS:
    # A. Select Top K
    subset = full_results[:k]

    # B. Re-sort by document flow (critical for readability/coherence)
    subset.sort(key=lambda x: x['node_index'])

    # C. Generate Summary Text
    text_summary = " ".join([s['text'] for s in subset])

    # D. Evaluate against Gold Long Summary
    scores = scorer.score(gold_text, text_summary)
    #r1 = scores['rouge1'].fmeasure
    rl = scores['rougeL'].fmeasure

    # E. Track Best Performance
    is_best = ""
    if rl > best_metric:
        best_metric = rl
        best_k = k
        is_best = "(*)" # Marker for current best

    print(f"{k:<5}   | {rl:.4f}   | {is_best}")

print("-" * 45)
print(f"✅ Optimal K found: {best_k} (ROUGE-L: {best_metric:.4f})")

# 5. Final Selection using Best K
selected_items = full_results[:best_k]
selected_items.sort(key=lambda x: x['node_index'])

# 6. Save to JSON (Strict Structure)
output_data = {
    "input_file": INPUT_FILE,
    "method": "gcn_regression_dynamic_k",
    "parameters": {
        "best_k": best_k,
        "candidate_k_range": CANDIDATE_KS,
        "gold_file_used": GOLD_SUMMARY_FILE,
        "best_rouge_l": best_metric
    },
    "selected": selected_items
}

out_filename = "long_summary_output.json"
with open(out_filename, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"Long summary saved to: {out_filename}")


--- Evaluating Candidate K Values (Long Summary) ---
K     | ROUGE-L  | Status
---------------------------------------------
10      | 0.1690   | (*)
15      | 0.1645   | 
20      | 0.1506   | 
25      | 0.1508   | 
30      | 0.1427   | 
35      | 0.1388   | 
40      | 0.1317   | 
45      | 0.1276   | 
50      | 0.1277   | 
---------------------------------------------
✅ Optimal K found: 10 (ROUGE-L: 0.1690)
Long summary saved to: long_summary_output.json


comparing summaries using diff k values

In [ ]:
!pip install bert-score rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
import torch
from rouge_score import rouge_scorer
from bert_score import score as bert_score_func
import numpy as np
import pandas as pd

# --- CONFIGURATION ---
GOLD_SUMMARY_FILE = "gold_summary_short.txt"
CANDIDATE_KS = [3, 5, 7, 10]

# 1. Load Gold Summary
print(f"Loading Short Gold Summary: {GOLD_SUMMARY_FILE}")
with open(GOLD_SUMMARY_FILE, "r", encoding="utf-8") as f:
    gold_text = f.read()

# 2. Get Model Predictions
print("Running Inference...")
model.eval()
with torch.no_grad():
    # Get logits (if classification) or scores (if regression)
    output = model(x, edge_index)

    # If using BCEWithLogitsLoss (Binary Classif), apply sigmoid
    # If using MSELoss (Regression with Sigmoid output), it's already 0-1
    # Check if output is raw logits or probs. Assuming probs here:
    if output.min() < 0 or output.max() > 1: # Likely logits
        scores = torch.sigmoid(output).cpu().numpy().flatten()
    else:
        scores = output.cpu().numpy().flatten()

# 3. Prepare Candidates
candidates = []
for i, sc in enumerate(scores):
    candidates.append({
        "id": i,
        "text": doc_sentences[i],
        "score": float(sc)
    })

# Sort by Confidence (Descending)
candidates.sort(key=lambda x: x['score'], reverse=True)

# 4. Initialize Scorers
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
results_table = []

print(f"\n--- Comparing K Values for SHORT Summary ---")

# 5. Loop through K Values
for k in CANDIDATE_KS:
    # A. Select Top K
    selection = candidates[:k]

    # B. Sort by ID (restore flow)
    selection.sort(key=lambda x: x['id'])

    # C. Generate Summary
    generated_summary = " ".join([s['text'] for s in selection])

    # D. Calculate ROUGE
    rouge_scores = scorer.score(gold_text, generated_summary)
    r1 = rouge_scores['rouge1'].fmeasure
    r2 = rouge_scores['rouge2'].fmeasure
    rl = rouge_scores['rougeL'].fmeasure

    # E. Calculate BERTScore
    # (P, R, F1) returns tensors, we take the F1 mean
    P, R, F1 = bert_score_func([generated_summary], [gold_text], lang="en", verbose=False)
    bert_f1 = F1.mean().item()

    # Store
    results_table.append({
        "K": k,
        "R1": r1,
        "R2": r2,
        "RL": rl,
        "BERT_F1": bert_f1
    })

# 6. Display Results
df_results = pd.DataFrame(results_table)
print(df_results.to_string(index=False, float_format="%.4f"))

# 7. Identify Best K (based on ROUGE-L)
best_row = df_results.loc[df_results['RL'].idxmax()]
print("\n" + "-"*30)
print(f"✅ Best Short K: {int(best_row['K'])}")
print(f"   ROUGE-L: {best_row['RL']:.4f} | BERT-F1: {best_row['BERT_F1']:.4f}")
print("-" * 30)

Loading Short Gold Summary: gold_summary_short.txt
Running Inference...

--- Comparing K Values for SHORT Summary ---


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 K     R1     R2     RL  BERT_F1
 3 0.2768 0.0432 0.1289   0.8185
 5 0.3013 0.0681 0.1356   0.8248
 7 0.2815 0.0764 0.1358   0.8220
10 0.2510 0.0711 0.1201   0.8253

------------------------------
✅ Best Short K: 7
   ROUGE-L: 0.1358 | BERT-F1: 0.8220
------------------------------


In [ ]:
import torch
from rouge_score import rouge_scorer
from bert_score import score as bert_score_func
import numpy as np
import pandas as pd

# --- CONFIGURATION ---
GOLD_SUMMARY_FILE = "gold_summary_long.txt"
# Wider range for long summaries
CANDIDATE_KS = [10, 15, 20, 25, 30, 35, 40, 45, 50]

# 1. Load Gold Summary
print(f"Loading Long Gold Summary: {GOLD_SUMMARY_FILE}")
with open(GOLD_SUMMARY_FILE, "r", encoding="utf-8") as f:
    gold_text = f.read()

# 2. Get Model Predictions
print("Running Inference...")
model.eval()
with torch.no_grad():
    output = model(x, edge_index)
    if output.min() < 0 or output.max() > 1:
        scores = torch.sigmoid(output).cpu().numpy().flatten()
    else:
        scores = output.cpu().numpy().flatten()

# 3. Prepare Candidates
candidates = []
for i, sc in enumerate(scores):
    candidates.append({
        "id": i,
        "text": doc_sentences[i],
        "score": float(sc)
    })

# Sort by Confidence (Descending)
candidates.sort(key=lambda x: x['score'], reverse=True)

# 4. Initialize Scorers
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
results_table = []

print(f"\n--- Comparing K Values for LONG Summary ---")

# 5. Loop through K Values
for k in CANDIDATE_KS:
    # A. Select Top K
    selection = candidates[:k]

    # B. Sort by ID (restore flow)
    selection.sort(key=lambda x: x['id'])

    # C. Generate Summary
    generated_summary = " ".join([s['text'] for s in selection])

    # D. Calculate ROUGE
    rouge_scores = scorer.score(gold_text, generated_summary)
    r1 = rouge_scores['rouge1'].fmeasure
    r2 = rouge_scores['rouge2'].fmeasure
    rl = rouge_scores['rougeL'].fmeasure

    # E. Calculate BERTScore
    # Note: Using batch size 1 here. For large docs, this might take a moment.
    P, R, F1 = bert_score_func([generated_summary], [gold_text], lang="en", verbose=False)
    bert_f1 = F1.mean().item()

    # Store
    results_table.append({
        "K": k,
        "R1": r1,
        "R2": r2,
        "RL": rl,
        "BERT_F1": bert_f1
    })

# 6. Display Results
df_results = pd.DataFrame(results_table)
print(df_results.to_string(index=False, float_format="%.4f"))

# 7. Identify Best K (based on ROUGE-L)
best_row = df_results.loc[df_results['RL'].idxmax()]
print("\n" + "-"*30)
print(f"✅ Best Long K: {int(best_row['K'])}")
print(f"   ROUGE-L: {best_row['RL']:.4f} | BERT-F1: {best_row['BERT_F1']:.4f}")
print("-" * 30)

Loading Long Gold Summary: gold_summary_long.txt
Running Inference...

--- Comparing K Values for LONG Summary ---


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


 K     R1     R2     RL  BERT_F1
10 0.4399 0.1102 0.1690   0.8239
15 0.4027 0.1047 0.1645   0.8253
20 0.3722 0.1138 0.1506   0.8301
25 0.3476 0.1177 0.1508   0.8322
30 0.3361 0.1228 0.1427   0.8359
35 0.3229 0.1191 0.1388   0.8406
40 0.3104 0.1145 0.1317   0.8406
45 0.3069 0.1219 0.1276   0.8422
50 0.2977 0.1168 0.1277   0.8437

------------------------------
✅ Best Long K: 10
   ROUGE-L: 0.1690 | BERT-F1: 0.8239
------------------------------


using bart

In [ ]:
import torch
from transformers import pipeline
from rouge_score import rouge_scorer
from bert_score import score as bert_score_func
import pandas as pd

# --- CONFIGURATION ---
GOLD_SUMMARY_FILE = "gold_summary_short.txt"
CANDIDATE_KS = [3, 5, 7, 10] # Small K for short summaries

# 1. Load Gold Summary
with open(GOLD_SUMMARY_FILE, "r", encoding="utf-8") as f:
    gold_text = f.read()

# 2. Initialize BART Summarizer
# We use 'facebook/bart-large-cnn' which is SOTA for abstractive summarization
print("Loading BART Model...")
device_idx = 0 if torch.cuda.is_available() else -1
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=device_idx)

# 3. Get GCN Extractive Scores
model.eval()
with torch.no_grad():
    scores_pred = model(x, edge_index)
    if scores_pred.min() < 0 or scores_pred.max() > 1:
        scores_pred = torch.sigmoid(scores_pred)
    scores = scores_pred.cpu().numpy().flatten()

# Prepare Candidates
candidates = []
for i, sc in enumerate(scores):
    candidates.append({"id": i, "text": doc_sentences[i], "score": float(sc)})
candidates.sort(key=lambda x: x['score'], reverse=True)

# 4. Evaluation Loop
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
results_table = []
final_summaries = {}

print(f"\n--- Hybrid GCN + BART Evaluation (SHORT) ---")

for k in CANDIDATE_KS:
    # A. Extractive Step (GCN)
    selection = candidates[:k]
    selection.sort(key=lambda x: x['id']) # Re-order for flow
    extractive_text = " ".join([s['text'] for s in selection])

    # B. Abstractive Step (BART)
    # Truncation is vital here as GCN might send too much text
    # max_length=100 ensures the output remains a "Short" summary
    try:
        bart_output = summarizer(
            extractive_text,
            max_length=100,
            min_length=30,
            do_sample=False,
            truncation=True
        )
        abstractive_summary = bart_output[0]['summary_text']
    except Exception as e:
        print(f"Error at K={k}: {e}")
        continue

    # C. Calculate Metrics (BART Output vs Gold)
    # ROUGE
    r_scores = scorer.score(gold_text, abstractive_summary)

    # BERTScore
    P, R, F1 = bert_score_func([abstractive_summary], [gold_text], lang="en", verbose=False)

    results_table.append({
        "K_Sentences": k,
        "ROUGE-1": r_scores['rouge1'].fmeasure,
        "ROUGE-2": r_scores['rouge2'].fmeasure,
        "ROUGE-L": r_scores['rougeL'].fmeasure,
        "BERT-F1": F1.mean().item()
    })

    final_summaries[k] = abstractive_summary

# 5. Display & Save
df_results = pd.DataFrame(results_table)
print(df_results.to_string(index=False, float_format="%.4f"))

# Save Best Result
best_k = int(df_results.loc[df_results['ROUGE-L'].idxmax()]['K_Sentences'])
import json
output = {
    "method": "hybrid_gcn_bart_short",
    "best_k_input": best_k,
    "metrics": df_results[df_results['K_Sentences'] == best_k].to_dict('records')[0],
    "generated_summary": final_summaries[best_k]
}

with open("hybrid_short_summary.json", "w") as f:
    json.dump(output, f, indent=4)
print(f"\nBest Short Summary (K={best_k}) saved to JSON.")

Loading BART Model...


config.json: 0.00B [00:00, ?B/s]

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [ ]:
import torch
from transformers import pipeline
from rouge_score import rouge_scorer
from bert_score import score as bert_score_func
import pandas as pd
import json
import pickle
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
INPUT_FILE = "case_1_graph.pkl"
GOLD_SUMMARY_FILE = "gold_summary_long.txt"
EMBED_MODEL = "all-MiniLM-L6-v2"
CANDIDATE_KS = [10, 15, 20, 25, 30, 35, 40, 45, 50]
BART_PARAMS = {"max_length": 400, "min_length": 150}

print("⚠️ FORCING CPU MODE due to GPU error...")
device = torch.device('cpu')

# 1. Re-Load Data (Safe from GPU corruption)
print(f"Loading Graph: {INPUT_FILE}")
with open(INPUT_FILE, "rb") as f:
    G = pickle.load(f)

nodes = list(G.nodes())
node_to_idx = {n: i for i, n in enumerate(nodes)}
doc_sentences = [str(G.nodes[n].get("text", "")) for n in nodes]

print(f"Loading Gold Summary: {GOLD_SUMMARY_FILE}")
with open(GOLD_SUMMARY_FILE, "r", encoding="utf-8") as f:
    gold_text = f.read()

# 2. Re-Create Embeddings & Graph on CPU
print("Computing Embeddings on CPU...")
embedder = SentenceTransformer(EMBED_MODEL, device='cpu')
x_np = embedder.encode(doc_sentences, convert_to_numpy=True)
x = torch.tensor(x_np, dtype=torch.float32).to(device)

src, dst = [], []
for u, v in G.edges():
    src.append(node_to_idx[u])
    dst.append(node_to_idx[v])
edge_index = torch.tensor([src, dst], dtype=torch.long).to(device)

# 3. Re-Define Model (To ensure it's on CPU)
class SupervisedGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=128):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)
        self.head = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        return torch.sigmoid(self.head(x))

model = SupervisedGCN(in_channels=x.shape[1]).to(device)
model.eval()

# 4. Get GCN Scores
print("Running GCN Inference on CPU...")
with torch.no_grad():
    scores_pred = model(x, edge_index)
    scores = scores_pred.numpy().flatten()

# Prepare Candidates
candidates = []
for i, sc in enumerate(scores):
    candidates.append({"id": i, "text": doc_sentences[i], "score": float(sc)})
candidates.sort(key=lambda x: x['score'], reverse=True)

# 5. Load BART on CPU (device=-1)
print("Loading BART on CPU (This may take a moment)...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)

# 6. Evaluation Loop
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
results_table = []
final_summaries = {}

print(f"\n--- Hybrid GCN + BART Evaluation (LONG - CPU MODE) ---")
print(f"{'K':<5} | {'R-1':<7} | {'R-2':<7} | {'R-L':<7} | {'BERT-F1'}")
print("-" * 50)

for k in CANDIDATE_KS:
    if len(candidates) == 0: break

    # A. Extractive
    selection = candidates[:k]
    selection.sort(key=lambda x: x['id'])
    extractive_text = " ".join([s['text'] for s in selection])

    # B. Abstractive
    try:
        bart_output = summarizer(
            extractive_text,
            do_sample=False,
            truncation=True,
            **BART_PARAMS
        )
        abstractive_summary = bart_output[0]['summary_text']
    except Exception as e:
        print(f"Error at K={k}: {e}")
        continue

    # C. Scoring
    r_scores = scorer.score(gold_text, abstractive_summary)

    # BERTScore on CPU
    try:
        P, R, F1 = bert_score_func([abstractive_summary], [gold_text], lang="en", verbose=False, device='cpu')
        bert_f1 = F1.mean().item()
    except Exception:
        bert_f1 = 0.0

    r1 = r_scores['rouge1'].fmeasure
    r2 = r_scores['rouge2'].fmeasure
    rl = r_scores['rougeL'].fmeasure

    print(f"{k:<5} | {r1:.4f}  | {r2:.4f}  | {rl:.4f}  | {bert_f1:.4f}")

    results_table.append({
        "K_Sentences": k,
        "ROUGE-1": r1,
        "ROUGE-2": r2,
        "ROUGE-L": rl,
        "BERT-F1": bert_f1
    })
    final_summaries[k] = abstractive_summary

# 7. Save Results
if results_table:
    df_results = pd.DataFrame(results_table)
    best_k = int(df_results.loc[df_results['ROUGE-L'].idxmax()]['K_Sentences'])

    output = {
        "method": "hybrid_gcn_bart_long_cpu",
        "best_k_input": best_k,
        "metrics": df_results[df_results['K_Sentences'] == best_k].to_dict('records')[0],
        "generated_summary": final_summaries[best_k]
    }

    with open("hybrid_long_summary.json", "w") as f:
        json.dump(output, f, indent=4)
    print(f"\n✅ Best Long Summary (K={best_k}) saved to hybrid_long_summary.json.")
else:
    print("\n❌ No results generated.")

In [ ]:
import torch
from transformers import pipeline
from rouge_score import rouge_scorer
from bert_score import score as bert_score_func
import pandas as pd
import json
import pickle
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm
from sentence_transformers import SentenceTransformer
import math

# --- CONFIGURATION ---
INPUT_FILE = "case_1_graph.pkl"
GOLD_SUMMARY_FILE = "gold_summary_long.txt"
EMBED_MODEL = "all-MiniLM-L6-v2"
CANDIDATE_KS = [10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75]

# Chunking Parameters
CHUNK_SIZE = 700  # Number of words per chunk (Safe buffer for 1024 token limit)
BART_PARAMS = {
    "max_length": 300,  # Max length per chunk summary
    "min_length": 50,   # Min length per chunk summary
    "do_sample": False
}

print("⚠️ RUNNING WITH CHUNKING (CPU MODE)...")
device = torch.device('cpu')

# 1. Load Data
print(f"Loading Graph: {INPUT_FILE}")
with open(INPUT_FILE, "rb") as f:
    G = pickle.load(f)

nodes = list(G.nodes())
node_to_idx = {n: i for i, n in enumerate(nodes)}
doc_sentences = [str(G.nodes[n].get("text", "")) for n in nodes]

print(f"Loading Gold Summary: {GOLD_SUMMARY_FILE}")
with open(GOLD_SUMMARY_FILE, "r", encoding="utf-8") as f:
    gold_text = f.read()

# 2. Embeddings & Graph (CPU)
print("Computing Embeddings...")
embedder = SentenceTransformer(EMBED_MODEL, device='cpu')
x_np = embedder.encode(doc_sentences, convert_to_numpy=True)
x = torch.tensor(x_np, dtype=torch.float32).to(device)

src, dst = [], []
for u, v in G.edges():
    src.append(node_to_idx[u])
    dst.append(node_to_idx[v])
edge_index = torch.tensor([src, dst], dtype=torch.long).to(device)

# 3. Model Definition
class SupervisedGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=128):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)
        self.head = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        return torch.sigmoid(self.head(x))

model = SupervisedGCN(in_channels=x.shape[1]).to(device)
model.eval()

# 4. GCN Inference
print("Running GCN Inference...")
with torch.no_grad():
    scores_pred = model(x, edge_index)
    scores = scores_pred.numpy().flatten()

candidates = []
for i, sc in enumerate(scores):
    candidates.append({"id": i, "text": doc_sentences[i], "score": float(sc)})
candidates.sort(key=lambda x: x['score'], reverse=True)

# 5. Load BART
print("Loading BART Summarizer...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)

# --- CHUNKING HELPER FUNCTION ---
from transformers import AutoTokenizer

# Load tokenizer explicitly (Same as model)
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

def chunk_and_summarize(text, chunk_size, summarizer, params):
    """
    Safely splits text into chunks based on TOKEN COUNT (not words).
    Ensures every chunk is strictly within the model's limit (1024).
    """
    # 1. Tokenize the entire text (return tensors)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    input_ids = inputs["input_ids"][0]

    # 2. Define strict limits
    # BART limit is 1024. We use 900 to leave room for special tokens [BOS]/[EOS]
    MAX_CHUNK_TOKENS = 900

    # 3. Split into chunks
    total_tokens = len(input_ids)
    chunk_summaries = []

    # print(f"  -> Input Tokens: {total_tokens}. Chunking...")

    for i in range(0, total_tokens, MAX_CHUNK_TOKENS):
        # Slice the tensor
        chunk_ids = input_ids[i : i + MAX_CHUNK_TOKENS]

        # Decode back to text
        chunk_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)

        try:
            # Summarize this safe chunk
            # Note: We don't need 'truncation=True' here because we manually ensured safety
            output = summarizer(chunk_text, **params)
            summary = output[0]['summary_text']
            chunk_summaries.append(summary)
        except Exception as e:
            print(f"    [Warning: Chunk failed: {e}]")
            continue

    if not chunk_summaries:
        return ""

    return " ".join(chunk_summaries)


# 6. Evaluation Loop
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
results_table = []
final_summaries = {}

print(f"\n--- Hybrid GCN + BART (With Chunking) ---")
print(f"{'K':<5} | {'R-1':<7} | {'R-2':<7} | {'R-L':<7} | {'BERT-F1'}")
print("-" * 50)

for k in CANDIDATE_KS:
    if len(candidates) == 0: break

    # A. Extractive Selection
    selection = candidates[:k]
    selection.sort(key=lambda x: x['id'])
    extractive_text = " ".join([s['text'] for s in selection])

    # B. Abstractive Summarization with Chunking
    try:
        final_summary = chunk_and_summarize(
            extractive_text,
            CHUNK_SIZE,
            summarizer,
            BART_PARAMS
        )
    except Exception as e:
        print(f"Error at K={k}: {e}")
        continue

    # C. Scoring
    r_scores = scorer.score(gold_text, final_summary)

    try:
        P, R, F1 = bert_score_func([final_summary], [gold_text], lang="en", verbose=False, device='cpu')
        bert_f1 = F1.mean().item()
    except:
        bert_f1 = 0.0

    r1 = r_scores['rouge1'].fmeasure
    r2 = r_scores['rouge2'].fmeasure
    rl = r_scores['rougeL'].fmeasure

    print(f"{k:<5} | {r1:.4f}  | {r2:.4f}  | {rl:.4f}  | {bert_f1:.4f}")

    results_table.append({
        "K_Sentences": k,
        "ROUGE-1": r1,
        "ROUGE-2": r2,
        "ROUGE-L": rl,
        "BERT-F1": bert_f1
    })
    final_summaries[k] = final_summary

# 7. Save Results
if results_table:
    df_results = pd.DataFrame(results_table)
    best_k = int(df_results.loc[df_results['ROUGE-L'].idxmax()]['K_Sentences'])

    output = {
        "method": "hybrid_gcn_bart_chunking",
        "best_k_input": best_k,
        "metrics": df_results[df_results['K_Sentences'] == best_k].to_dict('records')[0],
        "generated_summary": final_summaries[best_k]
    }

    with open("hybrid_long_chunked_summary.json", "w") as f:
        json.dump(output, f, indent=4)
    print(f"\n✅ Best Chunked Summary (K={best_k}) saved to hybrid_long_chunked_summary.json.")
else:
    print("\n❌ No results generated.")

saving best k value outputs

In [ ]:
import pickle
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, BatchNorm
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
INPUT_FILE = "case_1_graph.pkl"
GOLD_SUMMARY_FILE = "gold_summary_short.txt"
OUTPUT_FILENAME = "short_summary_selection.json"
EMBED_MODEL = "all-MiniLM-L6-v2"
TARGET_K = 5  # We want exactly 5 sentences for the short summary

# --- 1. DATA LOADING ---
print(f"Loading Graph: {INPUT_FILE}")
with open(INPUT_FILE, "rb") as f:
    G = pickle.load(f)

# Create Mappings
nodes = list(G.nodes())
node_to_idx = {n: i for i, n in enumerate(nodes)}
idx_to_node = {i: n for i, n in enumerate(nodes)}
doc_sentences = [str(G.nodes[n].get("text", "")) for n in nodes]

print(f"Loading Short Gold Summary: {GOLD_SUMMARY_FILE}")
with open(GOLD_SUMMARY_FILE, "r", encoding="utf-8") as f:
    gold_text = f.read()

# --- 2. LABEL GENERATION (Binary Classification) ---
# We label the sentences that have the highest ROUGE-F1 overlap with the gold summary as "1" (Target).
print("Generating Binary Labels...")
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
sent_scores = []

for i, sent in enumerate(doc_sentences):
    # Use F-Measure for Short Summary (Balance of Precision & Recall)
    s = scorer.score(gold_text, sent)
    sent_scores.append((i, s['rougeL'].fmeasure))

# Pick Top K as "Positive" (1.0), others as "Negative" (0.0)
sent_scores.sort(key=lambda x: x[1], reverse=True)
top_k_indices = {i for i, _ in sent_scores[:TARGET_K]}

y_np = np.array([1.0 if i in top_k_indices else 0.0 for i in range(len(doc_sentences))])
y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

print(f"Labels Created: {int(y.sum().item())} Positives, {len(y) - int(y.sum().item())} Negatives.")

# --- 3. FEATURE & GRAPH PREPARATION ---
print("Computing Node Embeddings (Sentence-BERT)...")
embedder = SentenceTransformer(EMBED_MODEL)
x_np = embedder.encode(doc_sentences, convert_to_numpy=True)
x = torch.tensor(x_np, dtype=torch.float32)

# Build Edge Index
src, dst = [], []
for u, v in G.edges():
    if u in node_to_idx and v in node_to_idx:
        src.append(node_to_idx[u])
        dst.append(node_to_idx[v])
edge_index = torch.tensor([src, dst], dtype=torch.long)

# --- 4. GCN MODEL DEFINITION (ShortGCN) ---
class ShortGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=128):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)
        self.head = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.4, training=self.training) # Higher dropout for small target
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.4, training=self.training)
        return self.head(x) # Return logits (BCEWithLogitsLoss handles sigmoid)

# --- 5. TRAINING ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ShortGCN(in_channels=x.shape[1]).to(device)
x, edge_index, y = x.to(device), edge_index.to(device), y.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)
criterion = nn.BCEWithLogitsLoss()

print("Training Short GCN...")
model.train()
for epoch in range(400): # More epochs for convergence on small positive set
    optimizer.zero_grad()
    out = model(x, edge_index)
    loss = criterion(out, y)
    loss.backward()
    optimizer.step()

print("Training Complete.")

# --- 6. INFERENCE & SELECTION ---
model.eval()
with torch.no_grad():
    logits = model(x, edge_index)
    # Apply sigmoid to get probabilities (0.0 to 1.0)
    scores = torch.sigmoid(logits).cpu().numpy().flatten()

# Prepare Candidate List
candidates = []
for i, score in enumerate(scores):
    candidates.append({
        "node_index": i,
        "orig_node_id": idx_to_node[i],
        "score": float(score),
        "text": doc_sentences[i]
    })

# Select Top K (Strictly 5)
candidates.sort(key=lambda x: x['score'], reverse=True)
selected_k = candidates[:TARGET_K]

# Re-sort by document appearance (Node Index) for readability
selected_k.sort(key=lambda x: x['node_index'])

# --- 7. SAVE OUTPUT ---
output_data = {
    "input_file": INPUT_FILE,
    "method": "gcn_binary_classification",
    "parameters": {
        "k": TARGET_K,
        "model": "ShortGCN",
        "objective": "High Precision (Top-5)"
    },
    "selected": selected_k
}

with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"✅ Short Summary Output saved to: {OUTPUT_FILENAME}")

In [ ]:
import pickle
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv, BatchNorm
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
INPUT_FILE = "case_1_graph.pkl"
GOLD_SUMMARY_FILE = "gold_summary_long.txt"
OUTPUT_FILENAME = "long_summary_selection.json"
EMBED_MODEL = "all-MiniLM-L6-v2"
TARGET_K = 60  # We want a broad coverage of 60 sentences

# --- 1. DATA LOADING ---
print(f"Loading Graph: {INPUT_FILE}")
with open(INPUT_FILE, "rb") as f:
    G = pickle.load(f)

# Create Mappings
nodes = list(G.nodes())
node_to_idx = {n: i for i, n in enumerate(nodes)}
idx_to_node = {i: n for i, n in enumerate(nodes)}
doc_sentences = [str(G.nodes[n].get("text", "")) for n in nodes]

print(f"Loading Long Gold Summary: {GOLD_SUMMARY_FILE}")
with open(GOLD_SUMMARY_FILE, "r", encoding="utf-8") as f:
    gold_text = f.read()

# --- 2. LABEL GENERATION (Regression) ---
# We calculate a continuous score (0-1) based on ROUGE Recall.
print("Generating Regression Labels...")
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
scores = []

for sent in doc_sentences:
    # Use Recall for Long Summary (Focus on Coverage)
    s = scorer.score(gold_text, sent)
    scores.append(s['rougeL'].recall)

scores = np.array(scores)

# Normalize scores to 0-1 range for Sigmoid output
if scores.max() > 0:
    y_np = (scores - scores.min()) / (scores.max() - scores.min())
else:
    y_np = scores

y = torch.tensor(y_np, dtype=torch.float32).unsqueeze(1)

print(f"Labels Created. Mean Score: {y.mean():.4f}")

# --- 3. FEATURE & GRAPH PREPARATION ---
print("Computing Node Embeddings (Sentence-BERT)...")
embedder = SentenceTransformer(EMBED_MODEL)
x_np = embedder.encode(doc_sentences, convert_to_numpy=True)
x = torch.tensor(x_np, dtype=torch.float32)

# Build Edge Index
src, dst = [], []
for u, v in G.edges():
    if u in node_to_idx and v in node_to_idx:
        src.append(node_to_idx[u])
        dst.append(node_to_idx[v])
edge_index = torch.tensor([src, dst], dtype=torch.long)

# --- 4. GCN MODEL DEFINITION (LongGCN) ---
class LongGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=128):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.bn1 = BatchNorm(hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.bn2 = BatchNorm(hidden_channels)
        self.head = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training) # Standard dropout
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        return torch.sigmoid(self.head(x)) # Output 0-1 for regression

# --- 5. TRAINING ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LongGCN(in_channels=x.shape[1]).to(device)
x, edge_index, y = x.to(device), edge_index.to(device), y.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.MSELoss() # Mean Squared Error for continuous scores

print("Training Long GCN...")
model.train()
for epoch in range(300):
    optimizer.zero_grad()
    out = model(x, edge_index)
    loss = criterion(out, y)
    loss.backward()
    optimizer.step()

print("Training Complete.")

# --- 6. INFERENCE & SELECTION ---
model.eval()
with torch.no_grad():
    scores_pred = model(x, edge_index)
    # Output is already sigmoid (0-1) from model definition
    scores = scores_pred.cpu().numpy().flatten()

# Prepare Candidate List
candidates = []
for i, score in enumerate(scores):
    candidates.append({
        "node_index": i,
        "orig_node_id": idx_to_node[i],
        "score": float(score),
        "text": doc_sentences[i]
    })

# Select Top K (Strictly 60)
candidates.sort(key=lambda x: x['score'], reverse=True)
selected_k = candidates[:TARGET_K]

# Re-sort by document appearance (Node Index) for readability
selected_k.sort(key=lambda x: x['node_index'])

# --- 7. SAVE OUTPUT ---
output_data = {
    "input_file": INPUT_FILE,
    "method": "gcn_regression_selection",
    "parameters": {
        "k": TARGET_K,
        "model": "LongGCN",
        "objective": "High Coverage (Top-60)"
    },
    "selected": selected_k
}

with open(OUTPUT_FILENAME, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=4, ensure_ascii=False)

print(f"✅ Long Summary Output saved to: {OUTPUT_FILENAME}")